In [ ]:
import os
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import jax
import jax.numpy as jnp
import numpy as np
import mujoco
from mujoco import mjx
from mujoco_playground._src import mjx_env
from ml_collections import config_dict
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from IPython import display as ipython_display

from rlx.playground.ppo import PPOConfig, train, get_obs, normalize_obs

In [ ]:
# ── XML ───────────────────────────────────────────────────────────────────────

BALL_SIZE   = 0.15
BALL_HEIGHT = 0.8
BALL_X      = 0.35

_XML = f"""
<mujoco model="Humanoid and a ball">
  <option timestep="0.005" iterations="1" ls_iterations="4">
    <flag eulerdamp="disable"/>
  </option>
  <visual>
    <map force="0.1" zfar="30" znear="0.1"/>
    <rgba haze="0.15 0.25 0.35 1"/>
    <global offwidth="1280" offheight="720" elevation="-20" azimuth="120"/>
    <quality shadowsize="2048" offsamples="4"/>
  </visual>
  <statistic center="0 0 0.7" extent="4"/>
  <asset>
    <texture type="skybox" builtin="gradient" rgb1=".3 .5 .7" rgb2="0 0 0" width="32" height="512"/>
    <texture name="body" type="cube" builtin="flat" mark="cross" width="128" height="128"
             rgb1="0.8 0.6 0.4" rgb2="0.8 0.6 0.4" markrgb="1 1 1" random="0.01"/>
    <material name="body" texture="body" texuniform="true" rgba="0.8 0.6 .4 1"/>
    <texture name="texplane" type="2d" builtin="checker" rgb1=".4 .4 .4" rgb2=".6 .6 .6"
             width="512" height="512"/>
    <material name="MatPlane" reflectance="0.3" texture="texplane" texrepeat="1 1"
              texuniform="true" rgba=".7 .7 .7 1"/>
    <texture name="texball" type="cube" builtin="flat" mark="cross" width="128" height="128"
             rgb1="0.6 0.6 0.6" rgb2="0.6 0.6 0.6" markrgb="1 1 1"/>
    <material name="ball" texture="texball" texuniform="true" rgba=".1 .9 .1 1"/>
  </asset>
  <default>
    <default class="ball">
      <geom type="sphere" material="ball" size="{BALL_SIZE}" mass="0.045"
            friction="0.7 0.075 0.075" solref="0.02 1.0"/>
    </default>
    <motor ctrlrange="-1 1" ctrllimited="true"/>
    <default class="body">
      <geom type="capsule" condim="3" friction=".7" solimp=".9 .99 .003" solref=".015 1"
            material="body" contype="0" conaffinity="0"/>
      <default class="thigh"><geom size=".06"/></default>
      <default class="shin"><geom fromto="0 0 0 0 0 -.3" size=".049"/></default>
      <default class="foot">
        <geom size=".027"/>
        <default class="foot1"><geom fromto="-.07 -.01 0 .14 -.03 0"/></default>
        <default class="foot2"><geom fromto="-.07  .01 0 .14  .03 0"/></default>
      </default>
      <default class="arm_upper"><geom size=".04"/></default>
      <default class="arm_lower"><geom size=".031"/></default>
      <default class="hand"><geom type="sphere" size=".04"/></default>
      <joint type="hinge" damping=".2" stiffness="1" armature=".01" limited="true"
             solimplimit="0 .99 .01"/>
      <default class="joint_big">
        <joint damping="5" stiffness="10"/>
        <default class="hip_x"><joint range="-30 10"/></default>
        <default class="hip_z"><joint range="-60 35"/></default>
        <default class="hip_y"><joint axis="0 1 0" range="-150 20"/></default>
        <default class="joint_big_stiff"><joint stiffness="20"/></default>
      </default>
      <default class="knee"><joint pos="0 0 .02" axis="0 -1 0" range="-160 2"/></default>
      <default class="ankle">
        <joint range="-50 50"/>
        <default class="ankle_y"><joint pos="0 0 .08" axis="0 1 0" stiffness="6"/></default>
        <default class="ankle_x"><joint pos="0 0 .04" stiffness="3"/></default>
      </default>
      <default class="shoulder"><joint range="-85 60"/></default>
      <default class="elbow"><joint range="-100 50" stiffness="0"/></default>
    </default>
  </default>
  <worldbody>
    <light directional="true" diffuse=".8 .8 .8" pos="0 0 10" dir="0 0 -10"/>
    <geom name="floor" type="plane" size="5 5 .5" material="MatPlane" condim="3"/>
    <body name="torso" pos="0 0 1.282" childclass="body">
      <camera name="back" pos="-4.5 0 1" xyaxes="0 -1 0 1 0 3" mode="trackcom"/>
      <camera name="side" pos="0 -4.5 1" xyaxes="1 0 0 0 1 3" mode="trackcom"/>
      <freejoint name="root"/>
      <geom name="torso" fromto="0 -.07 0 0 .07 0" size=".07"/>
      <geom name="waist_upper" fromto="-.01 -.06 -.12 -.01 .06 -.12" size=".06"/>
      <body name="head" pos="0 0 .19"><geom name="head" type="sphere" size=".09"/></body>
      <body name="waist_lower" pos="-.01 0 -.26">
        <geom name="waist_lower" fromto="0 -.06 0 0 .06 0" size=".06"/>
        <joint name="abdomen_z" pos="0 0 .065" axis="0 0 1" range="-45 45" class="joint_big_stiff"/>
        <joint name="abdomen_y" pos="0 0 .065" axis="0 1 0" range="-75 30" class="joint_big"/>
        <body name="pelvis" pos="0 0 -.165">
          <joint name="abdomen_x" pos="0 0 .1" axis="1 0 0" range="-35 35" class="joint_big"/>
          <geom name="butt" fromto="-.02 -.07 0 -.02 .07 0" size=".09"/>
          <body name="thigh_right" pos="0 -.1 -.04">
            <joint name="hip_x_right" axis="1 0 0" class="hip_x"/>
            <joint name="hip_z_right" axis="0 0 1" class="hip_z"/>
            <joint name="hip_y_right" class="hip_y"/>
            <geom name="thigh_right" fromto="0 0 0 0 .01 -.34" class="thigh"/>
            <body name="shin_right" pos="0 .01 -.4">
              <joint name="knee_right" class="knee"/>
              <geom name="shin_right" class="shin"/>
              <body name="foot_right" pos="0 0 -.39">
                <joint name="ankle_y_right" class="ankle_y"/>
                <joint name="ankle_x_right" class="ankle_x" axis="1 0 .5"/>
                <geom name="foot1_right" class="foot1"/>
                <geom name="foot2_right" class="foot2"/>
              </body>
            </body>
          </body>
          <body name="thigh_left" pos="0 .1 -.04">
            <joint name="hip_x_left" axis="-1 0 0" class="hip_x"/>
            <joint name="hip_z_left" axis="0 0 -1" class="hip_z"/>
            <joint name="hip_y_left" class="hip_y"/>
            <geom name="thigh_left" fromto="0 0 0 0 -.01 -.34" class="thigh"/>
            <body name="shin_left" pos="0 -.01 -.4">
              <joint name="knee_left" class="knee"/>
              <geom name="shin_left" fromto="0 0 0 0 0 -.3" class="shin"/>
              <body name="foot_left" pos="0 0 -.39">
                <joint name="ankle_y_left" class="ankle_y"/>
                <joint name="ankle_x_left" class="ankle_x" axis="-1 0 -.5"/>
                <geom name="foot1_left" class="foot1"/>
                <geom name="foot2_left" class="foot2"/>
              </body>
            </body>
          </body>
        </body>
      </body>
      <body name="upper_arm_right" pos="0 -.17 .06">
        <joint name="shoulder1_right" axis="2 1 1"  class="shoulder"/>
        <joint name="shoulder2_right" axis="0 -1 1" class="shoulder"/>
        <geom name="upper_arm_right" fromto="0 0 0 .16 -.16 -.16" class="arm_upper"/>
        <body name="lower_arm_right" pos=".18 -.18 -.18">
          <joint name="elbow_right" axis="0 -1 1" class="elbow"/>
          <geom name="lower_arm_right" fromto=".01 .01 .01 .17 .17 .17" class="arm_lower"/>
          <body name="hand_right" pos=".18 .18 .18">
            <geom name="hand_right" zaxis="1 1 1" class="hand"/>
          </body>
        </body>
      </body>
      <body name="upper_arm_left" pos="0 .17 .06">
        <joint name="shoulder1_left" axis="-2 1 -1" class="shoulder"/>
        <joint name="shoulder2_left" axis="0 -1 -1" class="shoulder"/>
        <geom name="upper_arm_left" fromto="0 0 0 .16 .16 -.16" class="arm_upper"/>
        <body name="lower_arm_left" pos=".18 .18 -.18">
          <joint name="elbow_left" axis="0 -1 -1" class="elbow"/>
          <geom name="lower_arm_left" fromto=".01 -.01 .01 .17 -.17 .17" class="arm_lower"/>
          <body name="hand_left" pos=".18 -.18 .18">
            <geom name="hand_left" zaxis="1 -1 1" class="hand"/>
          </body>
        </body>
      </body>
    </body>
    <body name="ball" pos="{BALL_X} 0 {BALL_HEIGHT}"
          quat="0.632456 -0.632456 0.316228 0.316228">
      <freejoint/>
      <geom class="ball" name="ball"/>
    </body>
  </worldbody>
  <contact>
    <exclude body1="waist_lower" body2="thigh_right"/>
    <exclude body1="waist_lower" body2="thigh_left"/>
    <pair geom1="foot1_left"  geom2="floor"/>
    <pair geom1="foot1_right" geom2="floor"/>
    <pair geom1="foot2_left"  geom2="floor"/>
    <pair geom1="foot2_right" geom2="floor"/>
    <pair geom1="foot1_left"  geom2="ball"/>
    <pair geom1="foot2_left"  geom2="ball"/>
    <pair geom1="shin_left"   geom2="ball"/>
  </contact>
  <actuator>
    <motor name="abdomen_y"       gear="40"  joint="abdomen_y"/>
    <motor name="abdomen_z"       gear="40"  joint="abdomen_z"/>
    <motor name="abdomen_x"       gear="40"  joint="abdomen_x"/>
    <motor name="hip_x_right"     gear="40"  joint="hip_x_right"/>
    <motor name="hip_z_right"     gear="40"  joint="hip_z_right"/>
    <motor name="hip_y_right"     gear="120" joint="hip_y_right"/>
    <motor name="knee_right"      gear="80"  joint="knee_right"/>
    <motor name="ankle_x_right"   gear="20"  joint="ankle_x_right"/>
    <motor name="ankle_y_right"   gear="20"  joint="ankle_y_right"/>
    <motor name="hip_x_left"      gear="40"  joint="hip_x_left"/>
    <motor name="hip_z_left"      gear="40"  joint="hip_z_left"/>
    <motor name="hip_y_left"      gear="120" joint="hip_y_left"/>
    <motor name="knee_left"       gear="80"  joint="knee_left"/>
    <motor name="ankle_x_left"    gear="20"  joint="ankle_x_left"/>
    <motor name="ankle_y_left"    gear="20"  joint="ankle_y_left"/>
    <motor name="shoulder1_right" gear="20"  joint="shoulder1_right"/>
    <motor name="shoulder2_right" gear="20"  joint="shoulder2_right"/>
    <motor name="elbow_right"     gear="40"  joint="elbow_right"/>
    <motor name="shoulder1_left"  gear="20"  joint="shoulder1_left"/>
    <motor name="shoulder2_left"  gear="20"  joint="shoulder2_left"/>
    <motor name="elbow_left"      gear="40"  joint="elbow_left"/>
  </actuator>
</mujoco>
"""

# ── Helpers ───────────────────────────────────────────────────────────────────

def _mjx_step(mjx_model, data, action, n_substeps):
    """Apply action for n_substeps physics steps."""
    def single_step(data, _):
        data = data.replace(ctrl=action)
        return mjx.step(mjx_model, data), None
    return jax.lax.scan(single_step, data, (), n_substeps)[0]

# ── Environment ───────────────────────────────────────────────────────────────

def _foot_tricks_config():
    return config_dict.create(
        ctrl_dt=0.025,   # 5 substeps × 0.005 s
        sim_dt=0.005,
        impl="jax",
        nconmax=100,
        njmax=100,
    )


class FootTricksEnv(mjx_env.MjxEnv):
    """Humanoid ball-bouncing environment for mujoco_playground PPO."""

    def __init__(self, config=None, config_overrides=None):
        super().__init__(config or _foot_tricks_config(), config_overrides)
        self._mj_model = mujoco.MjModel.from_xml_string(_XML)
        self._mj_model.opt.solver = mujoco.mjtSolver.mjSOL_CG
        self._mj_model.opt.iterations = 6
        self._mj_model.opt.ls_iterations = 6
        self._mj_model.opt.timestep = self.sim_dt
        self._mjx_model = mjx.put_model(self._mj_model, impl=self._config.impl)
        self._foot_left_id = self._mj_model.body("foot_left").id
        self._ball_id = self._mj_model.body("ball").id

    def reset(self, rng: jax.Array) -> mjx_env.State:
        rng, rng1, rng2 = jax.random.split(rng, 3)
        data = mjx.make_data(self.mj_model, impl=self._config.impl)
        noise_scale = 1e-2
        n_humanoid_q = self.mj_model.nq - 7  # exclude ball freejoint
        noise_q = jax.random.uniform(rng1, (n_humanoid_q,), minval=-noise_scale, maxval=noise_scale)
        noise_v = jax.random.uniform(rng2, (n_humanoid_q - 1,), minval=-noise_scale, maxval=noise_scale)
        qpos = data.qpos.at[:n_humanoid_q].add(noise_q)
        qvel = data.qvel.at[:n_humanoid_q - 1].set(noise_v)
        data = data.replace(qpos=qpos, qvel=qvel)
        data = mjx.forward(self.mjx_model, data)

        ball_z = data.qpos[-5]  # ball z: qpos[-7:-4] = xyz of ball freejoint
        metrics = {
            "reward/ball":  jnp.zeros(()),
            "reward/ctrl":  jnp.zeros(()),
            "reward/alive": jnp.zeros(()),
            "bounces":      jnp.zeros(()),
        }
        info = {"rng": rng, "prev_ball_z": ball_z}
        obs = self._get_obs(data, info)
        reward, done = jnp.zeros(2)
        return mjx_env.State(data, obs, reward, done, metrics, info)

    def step(self, state: mjx_env.State, action: jax.Array) -> mjx_env.State:
        data = _mjx_step(self.mjx_model, state.data, action, self.n_substeps)

        ball_pos = data.qpos[-7:-4]         # xyz from ball freejoint
        ball_z   = ball_pos[2]
        foot_pos = data.xpos[self._foot_left_id]
        torso_z  = data.qpos[2]             # humanoid freejoint z

        alive = (
            (torso_z >= 1.0) & (torso_z <= 3.0) &
            (ball_z  >= 0.3) & (ball_z  <= 3.0)
        ).astype(jnp.float32)
        dist_foot_ball = jnp.sqrt(jnp.sum(jnp.square(ball_pos[:2] - foot_pos[:2])))
        alive = jnp.where(dist_foot_ball > 2.0, 0.0, alive)

        ctrl_cost = 0.1 * jnp.sum(jnp.square(action))
        ball_rwd  = 5.0 * (1.0 - jnp.abs(ball_z - 1.0) / 2.7) * (ball_z >= 0.3)
        foot_rwd  = 5.0 * jnp.clip(1.0 - dist_foot_ball / 2.0, 0.0, 1.0)
        alive_rwd = 5.0
        reward    = ball_rwd + foot_rwd + alive_rwd - ctrl_cost

        bounce_thr = BALL_HEIGHT - 0.05
        is_bounce  = (
            (state.info["prev_ball_z"] < bounce_thr) & (ball_z >= bounce_thr)
        ).astype(jnp.float32)

        metrics = {
            "reward/ball":  ball_rwd,
            "reward/ctrl":  -ctrl_cost,
            "reward/alive": alive_rwd * alive,
            "bounces":      is_bounce,
        }
        info = {**state.info, "prev_ball_z": ball_z}
        obs  = self._get_obs(data, info)
        done = 1.0 - alive
        return mjx_env.State(data, obs, reward, done, metrics, info)

    def _get_obs(self, data: mjx.Data, info: dict) -> dict:
        # qpos[2:]: position excl. global x,y  (33 dims)
        # qvel:     joint + freejoint velocities (33 dims)
        # xpos[1:]: world-frame body positions   (17 × 3 = 51 dims)
        # xmat[1:]: world-frame body orientations (17 × 9 = 153 dims)
        # qfrc_actuator: actuator forces          (33 dims)
        obs = jnp.concatenate([
            data.qpos[2:],
            data.qvel,
            data.xpos[1:].ravel(),
            data.xmat[1:].ravel(),
            data.qfrc_actuator,
        ])
        return {"state": obs, "privileged_state": obs}

    @property
    def xml_path(self) -> str:
        return ""

    @property
    def action_size(self) -> int:
        return self._mjx_model.nu

    @property
    def mj_model(self) -> mujoco.MjModel:
        return self._mj_model

    @property
    def mjx_model(self) -> mjx.Model:
        return self._mjx_model


# Smoke-test
env = FootTricksEnv()
s0  = jax.jit(env.reset)(jax.random.key(0))
s1  = jax.jit(env.step)(s0, jnp.zeros(env.action_size))
print(f"obs shape: {s0.obs['state'].shape}   action_size: {env.action_size}")

In [ ]:
cfg = PPOConfig(
    env_id="FootTricks",
    num_envs=2048,
    total_timesteps=200_000_000,
    num_steps=10,
    num_minibatches=32,
    update_epochs=8,
    learning_rate=3e-4,
    gamma=0.9,
    gae_lambda=0.95,
    clip_coef=0.3,
    ent_coef=1e-3,
    vf_coef=0.5,
    max_grad_norm=1.0,
    norm_adv=True,
    actor_hidden_sizes=[32, 32, 32, 32],
    critic_hidden_sizes=[256, 256, 256, 256, 256],
    policy_obs_key="state",
    value_obs_key="privileged_state",
    log_frequency=10,
    eval_frequency=100,
    eval_episodes=128,
    use_checkpointing=False,
    use_wandb=False,
    verbose=False,
    progress_bar=False,
    seed=42,
)

# ── Live plot ─────────────────────────────────────────────────────────────────
# Sub-reward keys are logged as eval/<metric_key> by ppo.train()

PLOT_KEYS = [
    # row 1 — task performance
    ("training/episode_reward", "Episode reward",   "C0"),
    ("eval/bounces",            "Bounces (eval)",   "C1"),
    ("time/sps",                "Steps / sec",      "C5"),
    # row 2 — PPO losses
    ("loss/policy",             "Policy loss",      "C2"),
    ("loss/value",              "Value loss",       "C3"),
    ("diagnostics/clip_frac",   "Clip fraction",    "C4"),
    # row 3 — sub-rewards
    ("eval/reward/ball",        "Ball reward",      "C6"),
    ("eval/reward/alive",       "Alive reward",     "C7"),
    ("eval/reward/ctrl",        "Ctrl cost",        "C8"),
]

fig, axes = plt.subplots(3, 3, figsize=(14, 9))
axes = axes.ravel()
for ax, (_, label, _c) in zip(axes, PLOT_KEYS):
    ax.set_title(label, fontsize=9)
    ax.set_xlabel("steps (M)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)
fig.tight_layout()

logs = []

def log_callback(log, step):
    logs.append({"step": step, **log})
    if len(logs) < 2:
        return
    steps_m = [d["step"] / 1e6 for d in logs]
    for ax, (key, label, color) in zip(axes, PLOT_KEYS):
        vals = [d.get(key) for d in logs]
        valid = [(s, v) for s, v in zip(steps_m, vals) if v is not None]
        if len(valid) < 2:
            continue
        xs, ys = zip(*valid)
        ax.cla()
        ax.plot(xs, ys, color=color, linewidth=1.2)
        ax.set_title(label, fontsize=9)
        ax.set_xlabel("steps (M)", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"FootTricks — {step/1e6:.1f} M / {cfg.total_timesteps/1e6:.0f} M steps", fontsize=11)
    fig.tight_layout()
    ipython_display.clear_output(wait=True)
    ipython_display.display(fig)

# ── Train ─────────────────────────────────────────────────────────────────────

model, optimizer, actor_stats, critic_stats = train(
    cfg,
    env_factory=FootTricksEnv,
    log_callback=log_callback,
)

# Final: save and close (prevents Jupyter from auto-displaying a second time)
plt.savefig("foot_tricks_training.png", dpi=120)
plt.close(fig)

In [ ]:
# ── Rollout with trained policy ───────────────────────────────────────────────

raw_env   = FootTricksEnv()
jit_reset = jax.jit(raw_env.reset)
jit_step  = jax.jit(raw_env.step)

@jax.jit
def policy_fn(obs, actor_stats):
    norm_a = normalize_obs(get_obs(obs, cfg.policy_obs_key), actor_stats)
    norm_c = normalize_obs(get_obs(obs, cfg.value_obs_key), critic_stats)
    dist, _ = model(norm_a, norm_c)
    return dist.mode  # deterministic

key   = jax.random.key(42)
state = jit_reset(key)
trajectory = [state]

for _ in range(1000):
    action = policy_fn(state.obs, actor_stats)
    state  = jit_step(state, action)
    trajectory.append(state)
    if float(state.done) > 0.5:
        break

total_bounces = sum(float(s.metrics["bounces"]) for s in trajectory)
total_reward  = sum(float(s.reward) for s in trajectory)
print(f"Episode length: {len(trajectory)} steps")
print(f"Total reward:   {total_reward:.1f}")
print(f"Ball bounces:   {total_bounces:.0f}")

# ── Render GIF ────────────────────────────────────────────────────────────────

render_every = 2
frames = raw_env.render(
    trajectory[::render_every], camera="side", height=480, width=640
)
fps = int(1.0 / raw_env.dt / render_every)

gif_path = "foot_tricks.gif"
imageio.mimsave(
    gif_path,
    [np.uint8(f) for f in frames],
    fps=fps,
    loop=0,
)
print(f"Saved → {gif_path}  ({len(frames)} frames @ {fps} fps)")

# Display inline (works in Jupyter)
from IPython.display import Image
Image(gif_path)